<a href="https://colab.research.google.com/github/Boran-Sert/SoftITO_Projler/blob/master/Python/21_10_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)


In [ ]:
import random
random.seed(42)

kisiler=["zıpırcan","tripcan","ilgisu","nazlıhan","berkay","sarı saruhan","heymen","aren","setenay"]

zamanlar = ["sabah","öğle","akşam","dün","bugün","hafta sonu"]

yer_e= ["parka","okula","eve","kütüphaneye","denize","pazara","bahçede"]

yer_de =["parkta","okulda","evde","kütüphanede","denizde","pazarda","bahçede"]

isler = [("kitap","okudu"),("çay","içti"),("top","oynadı"),("resim","yaptı")]

havalar = ["güzel","soğuk","sıcak","yağmurlu","günesli","rüzgarlı"]

sifatlar = ["mutlu","yorgun","neşeli","sessiz","heyecanlı"]


def cumle():
  k,z = random.choice(kisiler),random.choice(zamanlar)
  i= random.randrange(len(yer_e))
  nesne,fiil = random.choice(isler)
  kalip =random.randrange(5)
  if kalip == 0 : return f"{z} {k} {yer_e[i]} gitti"
  if kalip == 1 : return f"{z} {k} {yer_de[i]} {nesne} {fiil}"
  if kalip == 2 : return f"{z} hava çok {random.choice(havalar)}"
  if kalip == 3 : return f"{k} bugün çok {random.choice(sifatlar)}"
  return f"{z} {k} {yer_de[i]} {nesne} {fiil}"

metin = "\n".join(cumle() for _ in range(3000))
print("Metin uzunluğu:",len(metin))
print(metin[:900])

Metin uzunluğu: 85422
sabah tripcan pazarda top oynadı
öğle nazlıhan pazarda kitap okudu
bugün tripcan kütüphaneye gitti
öğle tripcan okulda kitap okudu
hafta sonu nazlıhan pazarda resim yaptı
bugün aren evde kitap okudu
akşam heymen evde çay içti
sabah sarı saruhan parka gitti
akşam sarı saruhan denize gitti
bugün aren parka gitti
akşam setenay bahçede top oynadı
hafta sonu nazlıhan parkta kitap okudu
sabah berkay bahçede gitti
akşam heymen kütüphanede top oynadı
akşam sarı saruhan okula gitti
bugün ilgisu pazarda çay içti
dün hava çok güzel
nazlıhan bugün çok neşeli
öğle tripcan denizde top oynadı
dün aren pazarda resim yaptı
öğle berkay okulda top oynadı
bugün heymen kütüphanede top oynadı
bugün ilgisu kütüphaneye gitti
tripcan bugün çok heyecanlı
dün tripcan kütüphanede resim yaptı
bugün berkay bahçede gitti
akşam setenay bahçede gitti
dün berkay okula gitti
bugün berkay bahçede çay içti
hafta sonu tripca


In [ ]:
harfler = sorted(set(metin))
SOZLUK = len(harfler)
harf2no = {u:i for i,u in enumerate(harfler)}
no2harf = {i:u for i,u in enumerate(harfler)}
def kodla(s):
  return[harf2no[h] for h in s ]

def coz(sayilar):
  return "".join(no2harf(i) for i in sayilar)

print("sözlük boyutu",SOZLUK)
print("harfler:","".join(harfler))

sözlük boyutu 30
harfler: 
 abcdefghiklmnoprstuvyzçöüğış


In [ ]:
veri= torch.tensor(kodla(metin))
n=int(0.9*len(veri))
egitim_veri=veri[:n]
dogrulama_veri=veri[n:]

In [ ]:
baglam=64
yıgın = 32

def yigin_al(bolum):
  d = egitim_veri if bolum == "egitim" else dogrulama_veri
  bas = torch.randint(len(d)-baglam -1 ,(yıgın,))
  x = torch.stack ([d[i:i+baglam] for i in bas])
  y = torch.stack([d[i+1:i+baglam+1] for i in bas])
  return x,y

x,y =yigin_al("egitim")

In [ ]:
from torch._prims_common import Dim
class MaskeliDikkat(nn.Module):
  def __init__(self,d_model,n_kafa):
    super().__init__()
    self.n_kafa = n_kafa
    self.d_k = d_model //n_kafa
    self.qkv = nn.Linear(d_model,3*d_model)
    self.cikis = nn.Linear(d_model,d_model)
    self.register_buffer("maske",torch.tril(torch.ones(baglam,baglam)))
  def forward(self,x):
    B,L,D =x.shape
    q,k,v = self.qkv(x).split(D,dim=2)
    q = q.view(B,L,self.n_kafa,self.d_k).transpose(1,2)
    k = k.view(B,L,self.n_kafa,self.d_k).transpose(1,2)
    v = v.view(B,L,self.n_kafa,self.d_k).transpose(1,2)
    skor = q@k.transpose(-2,-1)/math.sqrt(self.d_k)
    skor = skor.masked_fill(self.maske[:L,:L][None, None] == 0,float("-inf"))
    agirlik = F.softmax(skor,dim=-1)
    cikti = (agirlik @v).transpose(1,2).reshape(B,L,D)
    return self.cikis(cikti)


In [ ]:
class GPTBlok(nn.Module):
  def __init__ (self,d_model,n_kafa):
    super().__init__()
    self.norm1 = nn.LayerNorm(d_model)
    self.dikkat = MaskeliDikkat(d_model,n_kafa)
    self.norm2 = nn.LayerNorm(d_model)

    self.ffn = nn.Sequential(
        nn.Linear(d_model,4*d_model),
        nn.ReLU(),
        nn.Linear(4*d_model,d_model),
        nn.Dropout(0.5)
    )
  def forward(self ,x):
    x=x+self.dikkat(self.norm1(x))

    x=x+self.ffn(self.norm2(x))

    return x

class MiniGPT (nn.Module):
  def __init__(self,d_model=64,n_kafa=4,n_katman=2):
    super().__init__()
    self.harf_gomme =nn.Embedding(SOZLUK,d_model)
    self.konum_gomme =nn.Embedding(baglam,d_model)
    self.blok = nn.Sequential(*[GPTBlok(d_model,n_kafa) for _ in range(n_katman)])
    self.norm = nn.LayerNorm(d_model)
    self.cikis = nn.Linear(d_model,SOZLUK)

  def forward (self,x,hedef =None):
    B,L =x.shape
    h = self.harf_gomme(x)+self.konum_gomme(torch.arange(L))
    h= self.norm(self.blok(h))
    logits = self.cikis(h)
    if hedef is None:
      return logits
    loss = F.cross_entropy(logits.view(-1,SOZLUK),hedef.view(-1))
    return logits,loss
  @torch.no_grad()
  def uret(self, baslangic, n_harf, sicaklik=0.8):
      x = torch.tensor(kodla(baslangic)).unsqueeze(0)   # [L] -> [1, L]
      for _ in range(n_harf):
          x_kirp = x[:, -baglam:]
          logits = self(x_kirp)                          # hedef yokken sadece logits döner
          logits = logits[:, -1, :] / sicaklik
          olasilik = F.softmax(logits, dim=-1)
          sonraki = torch.multinomial(olasilik, num_samples=1)
          x = torch.cat((x, sonraki), dim=1)
      return coz(x[0].tolist())

In [ ]:
model =MiniGPT()


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-3)
egitim_kayip ,dogrulama_kayip ,adimlar = [],[],[]


for adim in range (1,100):
  model.train()
  x,y =yigin_al("egitim")

  _,kayip = model(x,y)
  optimizer.zero_grad()
  kayip.backward()
  optimizer.step()

  if adim %100 ==0:
    model.eval()
    with torch.no_grad():
      _,d_kayip=model(*yigin_al("dogrulama"))
      adimlar.append(adim)
      egitim_kayip.append(kayip.item())
      dogrulama_kayip.append(d_kayip.item())
      if adim %300 == 0:
        print(f"Adım {adim:4d} |egitim kaybı {kayip.item():.3f}| dogrulama kaybı {d_kayip.item():.3f}" )

In [ ]:
def coz(sayilar):
  return "".join(no2harf[i] for i in sayilar)

model.eval()
for basla in ["bugün","ilgisu","tripcan"]:
  print(model.uret(basla,150))
  print("--"*80)

bugün sesizesiz
akşazldedı bugün silgün
hava be have bugün çoye he çok ba gün hayittiçok ilün çok çok güzıreyne ok gizgilgüzeyok sesiçok k k çok yark resur
----------------------------------------------------------------------------------------------------------------------------------------------------------------
ilgisu parı
be bahen oyme çı
haftan sok sıhava sene bugizgün ha çok çok yok gitrgutareneyok güpark y gün çokudeton gi
aymu
akşedün çok sitru gün çokuletton 
----------------------------------------------------------------------------------------------------------------------------------------------------------------
tripcan okşamule oynu
de düpı
saha ittoka a okudeymuden yn bugün okay bugüzdu gün çoye yok kşerk çok çok gittilgünedüpcanargi
denu
öğlile bugün gün riçoktttr
----------------------------------------------------------------------------------------------------------------------------------------------------------------
